# Análise Exploratória dos Dados Inicial

In [1]:
import sys
from pathlib import Path

# Garante que a raiz do projeto está no sys.path
ROOT = Path().resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


## O que encontrar neste Notebook?

Este notebook tem como objetivo conduzir uma análise exploratória inicial dos dados (EDA) no contexto de serviços de telecomunicações, com foco na compreensão da estrutura, qualidade e comportamento das variáveis disponíveis.

A análise será organizada de forma estruturada por grupos de variáveis — como demográficas, geográficas, financeiras e de relacionamento — permitindo uma leitura mais coerente sob a ótica de negócio. Para cada grupo, serão realizadas análises univariadas e bivariadas em relação à variável alvo (churn), com o intuito de identificar padrões, distribuições, possíveis inconsistências e indícios preliminares de associação.

Adicionalmente, ao final do notebook, será conduzida uma análise multivariada com abordagem exploratória, incluindo o uso de modelos interpretáveis, como árvores de decisão, para capturar interações entre variáveis e aprofundar a identificação de possíveis drivers de churn.

Este material tem como finalidade não apenas descrever os dados, mas também gerar hipóteses analíticas e direcionar as próximas etapas do desenvolvimento de modelos preditivos.

# Entendimento do Negócio

O presente dataset refere-se a um cenário fictício de uma empresa de telecomunicações que oferece serviços de telefonia e internet para seus clientes. O principal objetivo é analisar o fenômeno de customer churn, ou seja, a evasão de clientes — quando um usuário deixa de utilizar os serviços da empresa.

A base de dados foi estruturada para capturar múltiplas dimensões do comportamento dos clientes, incluindo características demográficas, serviços contratados, informações de faturamento e tempo de relacionamento com a empresa. A variável alvo indica se o cliente cancelou ou não o serviço em um determinado período.

Do ponto de vista de negócio, a análise de churn é altamente estratégica, uma vez que a retenção de clientes tende a ser significativamente mais econômica do que a aquisição de novos. Nesse contexto, compreender os fatores que influenciam a evasão permite direcionar ações mais eficazes de retenção, como campanhas segmentadas, melhorias na experiência do cliente e ajustes na oferta de serviços.

Assim, este dataset possibilita a investigação de padrões de comportamento associados ao churn, bem como o desenvolvimento de modelos preditivos capazes de identificar clientes com maior propensão a cancelar os serviços.

# Importando as Bibliotecas

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# Pacotes Matemáticos
from scipy import stats


from sklearn.tree import DecisionTreeClassifier

# Params
plt.rcParams['figure.dpi'] = 400
pd.set_option('display.max_columns', None)


# Importando a Base

In [3]:
# Dataset obtido em https://www.kaggle.com/datasets/yeanzc/telco-customer-churn-ibm-dataset
df = pd.read_excel('../../data/raw/Telco_customer_churn.xlsx')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/Telco_customer_churn.xlsx'

# Funções Auxiliares

In [ ]:
# Funções utilitárias extraídas para src/utils
from experiments.utils.eda import freq_table, sanity_check, taxa_churn_categoria
from experiments.utils.plots import (
    boxplots_target_binaria,
    grid_cat_freq_event_rate,
    plot_univariate,
)
from experiments.utils.stats import (
    AnaliseIV,
    calculate_vif,
    chi2_analysis,
    classify_vif,
    cramers_v,
)


# Dicionário de Dados (Metadados)

A tabela abaixo descreve as variáveis disponíveis no dataset, incluindo tipo, categoria e interpretação no contexto de negócio.


| Variável            | Tipo        | Categoria              | Descrição                                                                 |
|--------------------|------------|-----------------------|--------------------------------------------------------------------------|
| CustomerID         | Categórica | Identificação         | Identificador único do cliente                                           |
| Count              | Numérica   | Técnica               | Variável constante (valor fixo = 1) utilizada para contagem              |
| Country            | Categórica | Geográfica            | País do cliente                                                          |
| State              | Categórica | Geográfica            | Estado do cliente                                                        |
| City               | Categórica | Geográfica            | Cidade do cliente                                                        |
| Zip Code           | Numérica   | Geográfica            | Código postal                                                            |
| Lat Long           | Categórica | Geográfica            | Coordenadas geográficas combinadas                                       |
| Latitude           | Numérica   | Geográfica            | Latitude                                                                 |
| Longitude          | Numérica   | Geográfica            | Longitude                                                                |
| Gender             | Categórica | Demográfica           | Gênero do cliente                                                        |
| Senior Citizen     | Categórica | Demográfica           | Indica se o cliente é idoso                                              |
| Partner            | Categórica | Demográfica           | Indica se possui parceiro(a)                                             |
| Dependents         | Categórica | Demográfica           | Indica se possui dependentes                                             |
| Tenure Months      | Numérica   | Relacionamento        | Tempo de permanência do cliente (meses)                                 |
| Phone Service      | Categórica | Serviços              | Possui serviço telefônico                                                |
| Multiple Lines     | Categórica | Serviços              | Possui múltiplas linhas                                                  |
| Internet Service   | Categórica | Serviços              | Tipo de serviço de internet                                              |
| Online Security    | Categórica | Serviços              | Serviço de segurança online                                              |
| Online Backup      | Categórica | Serviços              | Serviço de backup online                                                 |
| Device Protection  | Categórica | Serviços              | Proteção de dispositivos                                                 |
| Tech Support       | Categórica | Serviços              | Suporte técnico                                                          |
| Streaming TV       | Categórica | Serviços              | Serviço de streaming de TV                                               |
| Streaming Movies   | Categórica | Serviços              | Serviço de streaming de filmes                                           |
| Contract           | Categórica | Relacionamento        | Tipo de contrato                                                         |
| Paperless Billing  | Categórica | Relacionamento        | Fatura digital                                                           |
| Payment Method     | Categórica | Relacionamento        | Método de pagamento                                                      |
| Monthly Charges    | Numérica   | Financeira            | Valor mensal cobrado                                                     |
| Total Charges      | Numérica*  | Financeira            | Valor total cobrado (pode exigir conversão de tipo)                     |
| Churn Label        | Categórica | Target                | Indica churn (Yes/No)                                                   |
| Churn Value        | Binária    | Target                | Indicador numérico de churn (1 = churn)                                 |
| Churn Score        | Numérica   | Score                 | Score preditivo de churn (derivado)                                     |
| CLTV               | Numérica   | Financeira/Score      | Lifetime value estimado do cliente                                      |
| Churn Reason       | Categórica | Pós-Churn          | Motivo do cancelamento (apenas para clientes churn)                     |

---

⚠️ **Notas importantes:**

- *Total Charges* está como string e deve ser convertido para numérico antes das análises.
- *Churn Score*, *CLTV* e *Churn Reason* podem introduzir **data leakage**, pois são derivados ou conhecidos após o evento de churn.
- *Churn Label* e *Churn Value* representam a mesma informação (usar apenas uma como target).

# Sanity Check

In [ ]:
df.info()

In [ ]:
sanity_check(df)

- A coluna `churn reason` possui 73,46% de dados faltantes. Isso é explicado por causa dessa variável ser responsável por armazenar a justificativa de abandono do cliente.
- Inconsistencias de dados foram encontradas nas colunas `Multiple Lines`, `Online Security`, `Online Backup`, `Device Protection`, `Text Support`, `Streaming TV`, `Streaming Movies`, `Payment Methods`
- Não temos dados duplicados, ou seja, cada linha do nosso dataframe representa um cliente.
- Coluna `count` é uma coluna criada para manipulação de filtros em dashboards e pode ser descartada.
- Coluna `Total Charges` está como object ao invés de float.

## Tratamento dos Dados

In [ ]:
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

In [ ]:
df['Total Charges'].info()

# Separando as variáveis

## Classificação das Variáveis

As variáveis do dataset foram organizadas em grupos de acordo com sua natureza e papel no contexto de negócio, permitindo uma análise exploratória estruturada e orientada à interpretação dos fatores associados ao churn.

### 🔹 Variáveis Geográficas
- Country
- State
- City
- Zip Code
- Latitude
- Longitude
- Lat Long

### 🔹 Variáveis Demográficas
- Gender
- Senior Citizen
- Partner
- Dependents

### 🔹 Variáveis de Relacionamento com o Cliente
- Tenure Months
- Contract
- Paperless Billing
- Payment Method

### 🔹 Variáveis de Serviços Contratados
- Phone Service
- Multiple Lines
- Internet Service
- Online Security
- Online Backup
- Device Protection
- Tech Support
- Streaming TV
- Streaming Movies

### 🔹 Variáveis Financeiras
- Monthly Charges
- Total Charges

### 🔹 Variáveis Derivadas / Score
- Churn Score
- CLTV

### 🔹 Variáveis Relacionadas ao Churn (Pós-Evento ⚠️)
- Churn Label
- Churn Value
- Churn Reason

### 🔹 Variáveis Técnicas / Identificação
- CustomerID
- Count

---

⚠️ **Observação importante:**  
Variáveis como *Churn Score*, *CLTV* e *Churn Reason* podem conter **informação derivada ou posterior ao evento de churn**, devendo ser utilizadas com cautela em análises preditivas para evitar *data leakage*.

In [ ]:
target = 'Churn Label'
num_vars = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
cat_vars = [col for col in df.columns if df[col].dtype in ['str', 'object', 'category']]

## Variáveis Numéricas

In [ ]:
num_vars

In [ ]:
geo_vars = ['Zip Code', 'Latitude', 'Longitude']

In [ ]:
# removendo variáveis geofráficas que não serão utilizadas agoras
for col in geo_vars:
    num_vars.remove(col)

# removendo a coluna Count - utilizada apenas para manipulação de filtros em dashboards
num_vars.remove('Count')

In [ ]:
num_vars

## Variáveis Categóricas

In [ ]:
cat_vars

In [ ]:
# são colunas que não vão impactar nas análises nesse primeiro momento
drop_cols = ['CustomerID', 'Country', 'State', 'Lat Long', 'Churn Label', 'Churn Reason']

for col in drop_cols:
    cat_vars.remove(col)

In [ ]:
cat_vars

# Análise de Outliers

In [ ]:
import math

n_cols = 2
n_rows = math.ceil(len(num_vars) / n_cols)

fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(12, 3 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num_vars):
    sns.boxplot(x=df[col], ax=axes[i], color='skyblue')
    axes[i].set_title(f'Boxplot de {col}', fontsize=10)
    axes[i].grid(True, linestyle='--', alpha=0.5)

# Remove eixos vazios
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

# Variáveis Geográficas

In [ ]:
df.columns

In [ ]:
vars_geograficas = [
    'Country',
    'State',
    'City',
    'Zip Code',
    'Lat Long',
    'Latitude',
    'Longitude'
]

In [ ]:
df[vars_geograficas].nunique()

In [ ]:
df[['Country', 'State']].value_counts()

- O Dataset avalia serviços apenas do estado da Califórnia nos Estados Unidos. Isso pode imputar limitações e vieses ao modelo de predição de Churn.

### Tabelas de Frequência

In [ ]:
# frequência absoluta
freq_abs = df['City'].value_counts(dropna=False)

# frequência relativa (%)
freq_rel = df['City'].value_counts(normalize=True, dropna=False) * 100

# criando DataFrame consolidado
tabela_cidades = pd.DataFrame({
    'Cidade': freq_abs.index,
    'Frequência Absoluta': freq_abs.values,
    'Frequência Relativa (%)': freq_rel.values
})

# ordenando (garantia)
tabela_cidades = tabela_cidades.sort_values(by='Frequência Absoluta', ascending=False)

# frequência acumulada (%)
tabela_cidades['Frequência Acumulada (%)'] = tabela_cidades['Frequência Relativa (%)'].cumsum()

# top 10 cidades
top10_cidades = tabela_cidades.head(10)

# resetando índice
top10_cidades = top10_cidades.reset_index(drop=True)

top10_cidades

# arredondando as colunas de porcentagem para 2 casas decimais
top10_cidades.style.format({
    'Frequência Relativa (%)': '{:.2f}',
    'Frequência Acumulada (%)': '{:.2f}'
})

### Mapa de Calor - Taxa de Churn por Cidade

In [ ]:
# garantir tipos corretos
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# remover nulos
df_geo = df.dropna(subset=['Latitude', 'Longitude', 'Churn Value'])

In [ ]:
df_top_regions = (
    df_geo
    .groupby(['City'])
    .agg(
        total_clientes=('Churn Value', 'count'),
        churn_rate=('Churn Value', 'mean')
    )
    .query('total_clientes >= 30')
    .sort_values('churn_rate', ascending=False)
    .head(10)
)

df_top_regions

In [ ]:
from folium import Map
from folium.plugins import HeatMap

# base do mapa (centro médio)
mapa = Map(
    location=[df_geo['Latitude'].mean(), df_geo['Longitude'].mean()],
    zoom_start=5
)

# dados para heatmap
heat_data = [
    [row['Latitude'], row['Longitude'], row['Churn Value']]
    for _, row in df_geo.iterrows()
]

# adiciona heatmap
HeatMap(heat_data, radius=8).add_to(mapa)

mapa

In [ ]:
# =========================
# REGION MAPPING (4 CLASSES)
# =========================

NORCAL_COUNTIES = {
    "Alameda","Contra Costa","Marin","Napa","San Francisco","San Mateo",
    "Santa Clara","Solano","Sonoma","Sacramento","Yolo","Placer","El Dorado",
    "Nevada","Yuba","Sutter","Butte","Shasta","Humboldt","Mendocino","Lake",
    "Del Norte","Trinity","Tehama","Siskiyou","Modoc","Lassen","Plumas","Sierra","Alpine","Amador","Calaveras","Tuolumne"
}

SOCAL_COUNTIES = {
    "Los Angeles","Orange","San Diego","Ventura"
}

CENTRAL_COUNTIES = {
    "Fresno","Kern","Kings","Tulare","Madera",
    "San Joaquin","Stanislaus","Merced",
    "Monterey","San Luis Obispo","Santa Barbara","Santa Cruz"
}

DESERT_COUNTIES = {
    "Riverside","San Bernardino","Imperial","Inyo"
}


def classify_region_from_county(county):
    if county in NORCAL_COUNTIES:
        return "norcal"
    elif county in SOCAL_COUNTIES:
        return "socal"
    elif county in CENTRAL_COUNTIES:
        return "central"
    elif county in DESERT_COUNTIES:
        return "socal"  # desert incluído como socal
    else:
        return "other"

In [ ]:
df['City_enc'] = df['City'].apply(classify_region_from_county)

In [ ]:
df['City_enc'].value_counts()

In [ ]:
df.groupby(['City_enc']).agg(
        total_clientes=('Churn Value', 'count'),
        churn_rate=('Churn Value', 'mean')
    )

# Variáveis Demográficas e Familiares

In [ ]:
vars_demograficas = [
    'Gender',
    'Senior Citizen',
    'Partner',
    'Dependents'
]

In [ ]:
df[vars_demograficas].describe(include='object')

In [ ]:
for col in vars_demograficas:
    print(f"valores únicos em {col}: {df[col].unique()}")

In [ ]:
for col in vars_demograficas:
    print(f"\n📊 Tabela de frequência para {col}:\n")
    display(freq_table(df, col))

In [ ]:
for var in vars_demograficas:
    print(f'\n📊 Variável: {var}')
    display(taxa_churn_categoria(df, var))

In [ ]:
grid_cat_freq_event_rate(
    df,
    cat_vars=vars_demograficas,
    target='Churn Value',
    n_cols=2
)

# Tipos de Serviços


In [ ]:
vars_servicos = [
    'Phone Service',
    'Multiple Lines',
    'Internet Service',
    'Online Security',
    'Online Backup',
    'Device Protection',
    'Tech Support',
    'Streaming TV',
    'Streaming Movies'
]

In [ ]:
df[vars_servicos].describe(include='object')

In [ ]:
for col in vars_servicos:
    print(f"valores únicos em {col}: {df[col].unique()}")

In [ ]:
for col in vars_servicos:
    print(f"\n📊 Tabela de frequência para {col}:\n")
    display(freq_table(df, col))

In [ ]:
for var in vars_servicos:
    print(f'\n📊 Variável: {var}')
    display(taxa_churn_categoria(df, var))

In [ ]:
grid_cat_freq_event_rate(
    df,
    cat_vars=vars_servicos,
    target='Churn Value',
    n_cols=3
)

# Variáveis Financeiras

### Contrato e Faturamento

In [ ]:
vars_contrato_pagamento = [
    'Contract',
    'Paperless Billing',
    'Payment Method'
]

In [ ]:
df[vars_contrato_pagamento].describe(include='object')

In [ ]:
for col in vars_contrato_pagamento:
    print(f"valores únicos em {col}: {df[col].unique()}")

In [ ]:
for col in vars_contrato_pagamento:
    print(f"\n📊 Tabela de frequência para {col}:\n")
    display(freq_table(df, col))

In [ ]:
for var in vars_contrato_pagamento:
    print(f'\n📊 Variável: {var}')
    display(taxa_churn_categoria(df, var))

In [ ]:
grid_cat_freq_event_rate(
    df,
    cat_vars=vars_contrato_pagamento,
    target='Churn Value',
    n_cols=3
)

### Receita e Monetização

In [ ]:
vars_receita = [
    'Monthly Charges',
    'Total Charges',
    'Tenure Months'
]

In [ ]:
df[vars_receita].describe()

In [ ]:
# subplots num_vars
for col in vars_receita:
    plot_univariate(df, col)

In [ ]:
boxplots_target_binaria(df, 'Churn Label', vars_receita, n_cols=1)

# CLTV - Valor do Cliente

In [ ]:
df['CLTV'].describe()

In [ ]:
# ===============================
# Configuração de estilo
# ===============================
sns.set_style("whitegrid")

# ===============================
# Criação do subplot
# ===============================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ===============================
# 1. Histograma (distribuição)
# ===============================
sns.histplot(
    data=df,
    x='CLTV',
    bins=30,
    kde=True,
    ax=axes[0]
)

axes[0].set_title('Distribuição de CLTV')
axes[0].set_xlabel('CLTV')
axes[0].set_ylabel('Frequência')

# ===============================
# 2. Boxplot vs Churn
# ===============================
sns.boxplot(
    data=df,
    x='Churn Label',
    y='CLTV',
    hue='Churn Label',
    palette="Set2",
    ax=axes[1]
)

axes[1].set_title('CLTV vs Churn')
axes[1].set_xlabel('Churn Label')
axes[1].set_ylabel('CLTV')

# ===============================
# Ajuste final
# ===============================
plt.tight_layout()
plt.show()

# Churn Score

In [ ]:
df['Churn Score'].describe()

In [ ]:
# ===============================
# Configuração de estilo
# ===============================
sns.set_style("whitegrid")

# ===============================
# Criação do subplot
# ===============================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ===============================
# 1. Histograma (distribuição)
# ===============================
sns.histplot(
    data=df,
    x='Churn Score',
    bins=30,
    kde=True,
    ax=axes[0]
)

axes[0].set_title('Distribuição de Churn Score')
axes[0].set_xlabel('Churn Score')
axes[0].set_ylabel('Frequência')

# ===============================
# 2. Boxplot vs Churn
# ===============================
sns.boxplot(
    data=df,
    x='Churn Label',
    y='Churn Score',
    hue='Churn Label',
    palette="Set2",
    ax=axes[1]
)

axes[1].set_title('Churn Score vs Churn')
axes[1].set_xlabel('Churn Label')
axes[1].set_ylabel('Churn Score')

# ===============================
# Ajuste final
# ===============================
plt.tight_layout()
plt.show()

# Taxa de Churn

In [ ]:
df['Churn Label'].value_counts(normalize=True)

In [ ]:
# Estilo seaborn
sns.set_theme(style="whitegrid")

# Contagem da variável target
target_counts = df[target].value_counts().sort_index()
sizes = target_counts.values

# Mapeamento semântico da target
label_map = {
    'No': 'Não Churn',
    'Yes': 'Churn'
}

# Plot
plt.figure(figsize=(6, 6))
wedges, _, autotexts = plt.pie(
    sizes,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white'}
)

# Legenda informativa
legend_labels = [
    f'{label_map[label]}: {count} observações'
    for label, count in zip(target_counts.index, sizes)
]

plt.legend(
    wedges,
    legend_labels,
    title='Situação do Cliente',
    loc='center left',
    bbox_to_anchor=(1, 0.5)
)

plt.title('Taxa de Churn')
plt.tight_layout()
plt.show()

## Motivo do Churn

In [ ]:
## motivo de Churn
motivo_churn = df['Churn Reason'].value_counts(normalize=True) * 100
motivo_churn

# Análise de Correlação/Associação

## PairPlot

In [ ]:
df_pair = df[num_vars].copy()
df_pair['Churn'] = df[target]
df_pair.drop('Churn Value', axis=1, inplace=True)
df_pair

In [ ]:
sns.pairplot(df_pair, hue='Churn', kind='kde')

## Matriz de Correlação

In [ ]:
fig = plt.figure(figsize=(8,6))
sns.heatmap(df[num_vars].corr(),
            fmt=".2f",
            cmap='RdBu_r',
            vmin=-1, vmax=1,
            annot=True)

## Análise de Multicolinearidade - VIF

In [ ]:
vif_df = calculate_vif(df, num_vars)
vif_df = classify_vif(vif_df)
vif_df

## Information Value (IV)

In [ ]:
target = 'Churn Value'
df_iv = AnaliseIV(df[cat_vars + [target]], target=target, convention="good_over_bad")
df_iv.get_lista_iv()

In [ ]:
df_iv.get_bivariada('City')

In [ ]:
df_iv.get_bivariada('Contract')

## Qui-Quadrado + V de Cramer para variáveis categóricas

In [ ]:
resultado = chi2_analysis(df, cat_vars, target='Churn Value')

display(resultado)

# Árvore de Decisão para análise de importância de variáveis

## Preparando os Dados

### Variáveis Dummy

In [ ]:
cat_vars.remove('City')
num_vars.remove('Churn Score')

In [ ]:
dummy_vars = pd.get_dummies(df[cat_vars], drop_first=True, dtype=int)
dummy_vars.head()

### df_final

In [ ]:
df_final = pd.concat([df[num_vars], dummy_vars], axis=1)
df_final.head()

### Separando X e y

In [ ]:
# features
X = df_final.drop(columns=[target])

# target
y = df_final[target]

## Plotando a Árvore de Decisão

In [ ]:
exploratory_tree = DecisionTreeClassifier(
    #max_depth = 3,
    min_samples_leaf=50,
    random_state = 42
)


exploratory_tree.fit(X,y)

In [ ]:
# visualizando a arvore
from sklearn import tree
sns.reset_defaults()
fig = plt.figure(figsize=(25, 10))
feature_names = X.columns.to_list()
tree.plot_tree(exploratory_tree, feature_names=feature_names, filled=True, class_names=['Não', 'Sim'], fontsize=15);

plt.title('Classificação dos Casos de Churn\n usando a Arvore de Decisão', fontsize=20)
plt.show()

## Features Importance

In [ ]:
importances = np.asarray(exploratory_tree.feature_importances_).ravel()  # garante numpy 1D
features = list(X.columns)

df_imp = pd.DataFrame({
    "feature": features,
    "importance": importances
})


df_imp["importance"] = pd.to_numeric(df_imp["importance"], errors="coerce")
n_nulls = df_imp["importance"].isna().sum()
if n_nulls > 0:
    print(f"Atenção: {n_nulls} importances viraram NaN ao converter para numérico. Confira o modelo/importances.")


df_imp = df_imp.sort_values(by="importance", ascending=False, kind="quicksort").reset_index(drop=True)
df_imp

# Principais Drivers de Churn

Tier 1 - (core drivers)
- Tenure Months
- Contract
- Internet Service
- Online Security
- Tech Support

Tier 2 - (relevantes)
- Dependents
- Payment Method
- Online Backup
- Device Protection

Tier 3 - (Moderados)
- Monthly Charges
- Paperless Billing
- Partner
- Senior Citizen

Tier 4 - (Irrelevantes)
- Gender
- Phone Service
- Multiple Lines

# Principais Conclusões

- A atual taxa de Churn é de **26,6%**;
- A função de risco de churn é **decrescente no tempo**, ou seja, quanto mais o tempo passa, menor o risco de churn;
- Churn é sensível ao preço (monthly charge), mas não é linear. Preço alto, aumenta o churn. Porém, clientes antigos toleram preço alto, enquanto clientes novos não;
- O tipo de Contrato atua como um mitigador de churn. Contratos anuais tem taxas de churn muito inferiores quando comparadas a contratos mensais;
- Entre os principais motivos de churn, a má experiência com o suporte da empresa foi fator decisivo em, aproximadamente, 20% dos casos;
- A concorrência oferecer melhores valores e serviços, foi responsável por cerca de 35% dos motivos de churn;
- Cerca de 25% dos motivos de Churn, estão relacionados ao mismatch entre expecativa e entrega do serviço contratado;
- A variável `City`apresenta alta granularidade e precisa ser transformada via feature engineering para ser incluida no modelo;
- `Churn Score` e `CLTV` precisam ser avaliadas com cuidado sob risco de leakage;
- `Total Charges`, `Tenure Months`, `Monthly Charges`, `CLTV` e `Churn Score` tiveram um alto vif, resultando em alta multicolinearide;

# Sugestões para Feature Engineering

1. Clusterização das Cidades em Regiões para ajudar a reduzir incidencia de classes raras com um único cenário;
2. Cluster de Engajamento Digital (Scorer de Engajamento de Serviços);
3. Flag Ternure_Group para extratificação de clientes novos e antigos - churn não é linear no tempo;
4. Proxy de percepção de custo;
5. Codificação Ordinal na variável de Contrato seguindo critério monotônico;
6. Flag de estabilidade familiar identificando quais casos os clientes possuem dependentes ou planos familiares;

# Próximos Passos

- Preenchimento do ML Canvas do projeto;
- Elaboração do MVP (baseline);